## Downloads

In [1]:
#r "nuget: Plotly.NET.Interactive"
#r "nuget: Plotly.NET.CSharp"

using Plotly.NET;
using Plotly.NET.CSharp;
using System;
using System.Linq;

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Plotly.NET.CSharp, 0.13.0 Plotly.NET.Interactive, 5.0.0

Loading extensions from `C:\Users\Quinn Epstein\.nuget\packages\plotly.net.interactive\5.0.0\lib\netstandard2.1\Plotly.NET.Interactive.dll`

---

## Exercise 1: Exam Scores (N = 8)

### Questions:
1) my understanding is that ddof is a way to correct for sample size bias, by changing the way the varience/stdDev is calculated, by better accounting for the real spread of data

2) seeing as i did it manually, i did it by getting the value of the numbers at indexs q1 and q3, and subtracting the former from the latter.

3) the further away the median is from the mean the more large outliars there are on the side of the mean 

In [2]:
double[] scores = { 72, 85, 90, 68, 95, 78, 88, 84 };

double mean = scores.Average();
Console.WriteLine($"Mean Score: {mean}");

var sortedScores = scores.OrderBy(score => score).ToArray();
int n = sortedScores.Length;
double median = sortedScores[(n - 1) / 2] + (sortedScores[n / 2] - sortedScores[(n - 1) / 2]) / 2.0;
Console.WriteLine($"Median Score: {median}");

double range = scores.Max() - scores.Min();
Console.WriteLine($"Range of Scores: {range}");

double stdDev = Math.Sqrt(scores.Average(x => Math.Pow(x - mean, 2)));
System.Console.WriteLine($"Standard Deviation: {stdDev}");
System.Console.WriteLine($"Variance: {Math.Pow(stdDev, 2)}");

var lowHalf = sortedScores.Take(n / 2).ToArray();
var highHalf = sortedScores.Skip(n / 2).ToArray();
    
double q1 = lowHalf[(lowHalf.Length - 1) / 2] + (lowHalf[lowHalf.Length / 2] - lowHalf[(lowHalf.Length - 1) / 2]) / 2.0;
double q3 = highHalf[(highHalf.Length - 1) / 2] + (highHalf[highHalf.Length / 2] - highHalf[(highHalf.Length - 1) / 2]) / 2.0;
System.Console.WriteLine($"IQR: {q3 - q1}");

Mean Score: 82.5
Median Score: 84.5
Range of Scores: 27
Standard Deviation: 8.602325267042627
Variance: 74
IQR: 14


---
## Exercise 2: Factory Bolts & Coefficient of Variation

### Questions:
1) CV translates the stdDev across different circumstances, where the values dont match otherwise, by making it precentage based. 

2) A is much more consistant, as its CV is nearly 0

3) if CV is high, it means the product is not consistent, which is a nightmare for QC

In [3]:
double[] factoryA = { 50.1, 49.8, 50.3, 50.0, 49.9, 50.2 };
double[] factoryB = { 48.5, 51.2, 49.0, 51.8, 50.5, 49.3 };

// A quick local function to calculate all stats at once
// Notice we calculate Variance first, then Std Dev, just like we discussed!
(double Mean, double Var, double StdDev, double CV) CalculateStats(double[] data)
{
    double mean = data.Average();
    double variance = data.Average(x => Math.Pow(x - mean, 2)); // ddof=0 is baked in here
    double stdDev = Math.Sqrt(variance);
    double cv = (stdDev / mean) * 100;
    
    return (mean, variance, stdDev, cv);
}

// Run the data through our pipeline
var statsA = CalculateStats(factoryA);
var statsB = CalculateStats(factoryB);

// Print the Side-by-Side Table
// The ,-15 adds exact spacing to make the console output look like a perfect grid!
Console.WriteLine($"{"Metric",-15} | {"Factory A",-15} | {"Factory B",-15}");
Console.WriteLine(new string('-', 53));
Console.WriteLine($"{"Mean (μ)",-15} | {statsA.Mean,-15:F3} | {statsB.Mean,-15:F3}");
Console.WriteLine($"{"Variance (σ²)",-15} | {statsA.Var,-15:F3} | {statsB.Var,-15:F3}");
Console.WriteLine($"{"Std Dev (σ)",-15} | {statsA.StdDev,-15:F3} | {statsB.StdDev,-15:F3}");
Console.WriteLine($"{"CV (%)",-15} | {statsA.CV,-15:F3} | {statsB.CV,-15:F3}");

Metric          | Factory A       | Factory B      
-----------------------------------------------------
Mean (μ)        | 50.050          | 50.050         
Variance (σ²)   | 0.029           | 1.443          
Std Dev (σ)     | 0.171           | 1.201          
CV (%)          | 0.341           | 2.400          


---
## Exercise 3: Z-Scores to Identify Outliers

### Questions 
1) Z score shows us how far the data is from the average in standard deviations
2) because of the fact that 95% of data lies within 2 std devs its likely its an outlier or a bug
3) because deviance squares the values, the outlier can balloon it, even when its translated back into std dev, and since the average directly uses all the values, one value way out of line, will pull all the others far from where they should be

In [4]:
double[] temps = { 22.1, 21.8, 22.5, 31.2, 22.0, 21.5, 22.3, 21.9, 22.4, 21.7 };

// 1. Calculate Mean and Std Dev
double mean = temps.Average();
double variance = temps.Average(x => Math.Pow(x - mean, 2));
double stdDev = Math.Sqrt(variance);

Console.WriteLine($"Mean (μ): {mean:F2}°C");
Console.WriteLine($"Std Dev (σ): {stdDev:F2}°C\n");

// 2. Print Table Header
Console.WriteLine($"{"Day",-5} | {"Temp (°C)",-10} | {"Z-Score",-10} | {"Outlier?",-10}");
Console.WriteLine(new string('-', 46));

// 3. Calculate Z-Scores and Flag Outliers
for (int i = 0; i < temps.Length; i++)
{
    // The Z-Score formula
    double zScore = (temps[i] - mean) / stdDev;
    
    // Math.Abs() handles the |Z| absolute value requirement
    bool isOutlier = Math.Abs(zScore) > 2;
    string flag = isOutlier ? "YES" : "No";

    // i + 1 adjusts the index so Day starts at 1 instead of 0
    Console.WriteLine($"{i + 1,-5} | {temps[i],-10:F1} | {zScore,-10:F2} | {flag,-10}");
}

Mean (μ): 22.94°C
Std Dev (σ): 2.77°C

Day   | Temp (°C)  | Z-Score    | Outlier?  
----------------------------------------------
1     | 22.1       | -0.30      | No        
2     | 21.8       | -0.41      | No        
3     | 22.5       | -0.16      | No        
4     | 31.2       | 2.98       | YES       
5     | 22.0       | -0.34      | No        
6     | 21.5       | -0.52      | No        
7     | 22.3       | -0.23      | No        
8     | 21.9       | -0.38      | No        
9     | 22.4       | -0.19      | No        
10    | 21.7       | -0.45      | No        


---
## Exercise 4: World GDP per Capita

### Questions:

1)

2)

3)

In [5]:
// 1. Setup the Population Data
Dictionary<string, double> gdp = new Dictionary<string, double>
{
    {"Luxembourg", 126598}, {"Ireland", 99013}, {"Switzerland", 91932},
    {"Norway", 82236}, {"USA", 76330}, {"Denmark", 67803},
    {"Australia", 65099}, {"Netherlands", 62448}, {"Sweden", 55873},
    {"Germany", 52824}, {"Canada", 52722}, {"UK", 48866},
    {"France", 44408}, {"Japan", 33950}, {"South Korea", 32423},
    {"Spain", 30996}, {"Italy", 34085}, {"Brazil", 8920},
    {"Mexico", 13811}, {"India", 2485}
};

double[] popValues = gdp.Values.ToArray();
int N = popValues.Length;

// 2. Calculate Population Parameters (ddof = 0)
double mu = popValues.Average();
double popVariance = popValues.Average(x => Math.Pow(x - mu, 2)); 
double sigma = Math.Sqrt(popVariance);

// 3. Draw a Random Sample of n = 6
Random rnd = new Random();
double[] sampleValues = popValues.OrderBy(x => rnd.Next()).Take(6).ToArray();
int n = sampleValues.Length;

// 4. Calculate Sample Statistics (ddof = 1)
double xBar = sampleValues.Average();
// Using .Sum() instead of .Average() so we can manually divide by (n - 1)
double sampleVariance = sampleValues.Sum(x => Math.Pow(x - xBar, 2)) / (n - 1); 
double s = Math.Sqrt(sampleVariance);

// 5. Print the Comparison Table
Console.WriteLine($"{"Metric",-15} | {"Population (Param)",-20} | {"Sample (Stat)",-15}");
Console.WriteLine(new string('-', 56));
Console.WriteLine($"{"Mean",-15} | μ = {mu,-16:F0} | x̄ = {xBar,-11:F0}");
Console.WriteLine($"{"Std Dev",-15} | σ = {sigma,-16:F0} | s = {s,-11:F0}");

Metric          | Population (Param)   | Sample (Stat)  
--------------------------------------------------------
Mean            | μ = 54141            | x̄ = 48610      
Std Dev         | σ = 30525            | s = 26103      
